In [3]:
import torch
from sklearn.datasets import fetch_california_housing
housing = fetch_california_housing()

In [4]:
X = housing['data']
y = housing['target']

from sklearn.model_selection import train_test_split
X_train_full, X_test, y_train_full, y_test = train_test_split(X,y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full,y_train_full)

print(X_train.shape, X_test.shape, X_valid.shape)

from sklearn.preprocessing import StandardScaler
scl = StandardScaler()
scl.fit(X_train)

X_train = scl.transform(X_train)
X_test = scl.transform(X_test)
X_valid = scl.transform(X_valid)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
X_valid = torch.FloatTensor(X_valid)

y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)
y_valid = torch.tensor(y_valid, dtype=torch.float32).view(-1,1)

(11610, 8) (5160, 8) (3870, 8)


In [5]:
import torch.nn as nn

In [55]:
learning_rate = 0.1
n_epochs = 20

model = nn.Sequential(
	nn.Linear(in_features=8, out_features=3),
	nn.ReLU(),
	nn.Linear(in_features=3, out_features=1),
	)
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=learning_rate)

for epoch in range(n_epochs):
	y_pred = model(X_train)
	loss = criterion(y_pred, y_train)
	print(f"Epoch {epoch+1}/{n_epochs}, Loss: {round(loss.item(),3)}")
	loss.backward()
	optimizer.step()
	optimizer.zero_grad()

Epoch 1/20, Loss: 4.067
Epoch 2/20, Loss: 2.874
Epoch 3/20, Loss: 2.227
Epoch 4/20, Loss: 1.84
Epoch 5/20, Loss: 1.598
Epoch 6/20, Loss: 1.439
Epoch 7/20, Loss: 1.327
Epoch 8/20, Loss: 1.24
Epoch 9/20, Loss: 1.166
Epoch 10/20, Loss: 1.099
Epoch 11/20, Loss: 1.037
Epoch 12/20, Loss: 0.979
Epoch 13/20, Loss: 0.927
Epoch 14/20, Loss: 0.88
Epoch 15/20, Loss: 0.839
Epoch 16/20, Loss: 0.805
Epoch 17/20, Loss: 0.777
Epoch 18/20, Loss: 0.754
Epoch 19/20, Loss: 0.736
Epoch 20/20, Loss: 0.722


In [57]:
model = nn.Sequential(
	nn.Linear(in_features=8, out_features=3),
	nn.ReLU(),
	nn.Linear(in_features=3, out_features=1),
	)

In [80]:
w_son = model[2]._parameters['weight']
b_son = model[2]._parameters['bias']

In [65]:
w  = model[0]._parameters['weight']
b  = model[0]._parameters['bias']

w

Parameter containing:
tensor([[ 6.0728e-03,  1.8908e-01, -1.1398e-01,  3.0571e-01, -3.3858e-02,
         -2.9758e-01,  2.6366e-01,  2.9060e-01],
        [-1.3731e-01,  9.5968e-02, -1.5854e-01, -7.8499e-02,  3.4662e-01,
         -3.3364e-04, -1.0597e-01, -3.3622e-01],
        [-1.3445e-01,  1.9571e-01, -2.3887e-01, -5.5431e-02,  3.3795e-01,
         -1.1880e-01,  9.9714e-02,  3.0389e-01]], requires_grad=True)

In [68]:
X.shape, w.shape

(torch.Size([3, 8]), torch.Size([3, 8]))

In [70]:
X

tensor([[-0.3495,  0.6643, -0.2991, -0.0689, -0.1087, -0.0907, -1.3781,  1.1906],
        [ 0.4714,  0.5054,  0.2128, -0.2040, -0.0185,  0.0916, -0.8538,  0.8008],
        [ 0.1591, -0.2890, -0.2386, -0.1526, -0.3896,  0.0051, -0.5589, -0.1139]])

In [75]:
X @ w.T + b

tensor([[-0.1513, -0.3124,  0.4391],
        [-0.3083, -0.4042,  0.1254],
        [-0.5430, -0.2223, -0.2468]], grad_fn=<AddBackward0>)

In [76]:
z = model[0](X)
z

tensor([[-0.1513, -0.3124,  0.4391],
        [-0.3083, -0.4042,  0.1254],
        [-0.5430, -0.2223, -0.2468]], grad_fn=<AddmmBackward0>)

In [81]:
zz = model[1](z)
zz

tensor([[0.0000, 0.0000, 0.4391],
        [0.0000, 0.0000, 0.1254],
        [0.0000, 0.0000, 0.0000]], grad_fn=<ReluBackward0>)

In [82]:
w_son

Parameter containing:
tensor([[ 0.2227, -0.0007,  0.4283]], requires_grad=True)

In [84]:
zz @ w_son.T + b_son

tensor([[ 0.0869],
        [-0.0474],
        [-0.1011]], grad_fn=<AddBackward0>)

In [87]:
model[2](zz)

tensor([[ 0.0869],
        [-0.0474],
        [-0.1011]], grad_fn=<AddmmBackward0>)

In [89]:
y_pred = model(X)
y_pred

tensor([[ 0.0869],
        [-0.0474],
        [-0.1011]], grad_fn=<AddmmBackward0>)

In [94]:
((y_train[:3, :] - y_pred)**2).mean()

tensor(7.0338, grad_fn=<MeanBackward0>)

In [99]:
X_train.shape[0] // 32

362

In [101]:
X_train.shape[0] - 362 * 32

26

In [108]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [127]:
learning_rate = 0.1
n_epochs = 20

model = nn.Sequential(
	nn.Linear(in_features=8, out_features=3),
	nn.ReLU(),
	nn.Linear(in_features=3, out_features=1),
	).to('cpu') # cuda
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=learning_rate)

for epoch in range(n_epochs):
	total_loss = 0
	for X_batch, y_batch in train_loader:
		y_pred = model(X_batch)
		loss = criterion(y_pred, y_batch)
		total_loss += loss.item()
		loss.backward()
		optimizer.step()
		optimizer.zero_grad()
	avg_loss = total_loss / len(train_loader)
	print(f"Epoch {epoch+1}/{n_epochs}, Loss: {round(avg_loss,3)}")

Epoch 1/20, Loss: 0.61
Epoch 2/20, Loss: 0.744
Epoch 3/20, Loss: 0.478
Epoch 4/20, Loss: 0.451
Epoch 5/20, Loss: 0.497
Epoch 6/20, Loss: 0.456
Epoch 7/20, Loss: 0.423
Epoch 8/20, Loss: 0.438
Epoch 9/20, Loss: 0.47
Epoch 10/20, Loss: 0.441
Epoch 11/20, Loss: 0.42
Epoch 12/20, Loss: 0.417
Epoch 13/20, Loss: 0.412
Epoch 14/20, Loss: 0.441
Epoch 15/20, Loss: 0.439
Epoch 16/20, Loss: 0.421
Epoch 17/20, Loss: 0.421
Epoch 18/20, Loss: 0.423
Epoch 19/20, Loss: 0.42
Epoch 20/20, Loss: 0.419


In [ ]:
# !pip install torchmetrics

In [ ]:
import torchmetrics

In [139]:
learning_rate = 0.1
n_epochs = 20

model = nn.Sequential(
	nn.Linear(in_features=8, out_features=3),
	nn.ReLU(),
	nn.Linear(in_features=3, out_features=1),
	).to('cpu') # cuda
criterion = nn.MSELoss()
optimizer = torch.optim.SGD(params=model.parameters(), lr=learning_rate)
metric = torchmetrics.MeanAbsoluteError() # 'cuda'

for epoch in range(n_epochs):
	total_loss = 0
	metric.reset()
	for X_batch, y_batch in train_loader:
		y_pred = model(X_batch)
		loss = criterion(y_pred, y_batch)
		total_loss += loss.item()
		loss.backward()
		optimizer.step()
		optimizer.zero_grad()
		metric.update(y_pred, y_batch)
	avg_metric = metric.compute().item()
	avg_loss = total_loss / len(train_loader)
	print(f"Epoch {epoch+1}/{n_epochs}, Loss: {round(avg_loss,3)}, Metric: {round(avg_metric,3)}")

Epoch 1/20, Loss: 0.607, Metric: 0.556
Epoch 2/20, Loss: 0.481, Metric: 0.499
Epoch 3/20, Loss: 0.461, Metric: 0.486
Epoch 4/20, Loss: 0.463, Metric: 0.486
Epoch 5/20, Loss: 0.466, Metric: 0.49
Epoch 6/20, Loss: 0.524, Metric: 0.524
Epoch 7/20, Loss: 0.455, Metric: 0.488
Epoch 8/20, Loss: 0.437, Metric: 0.476
Epoch 9/20, Loss: 0.432, Metric: 0.474
Epoch 10/20, Loss: 0.43, Metric: 0.473
Epoch 11/20, Loss: 0.43, Metric: 0.473
Epoch 12/20, Loss: 0.424, Metric: 0.469
Epoch 13/20, Loss: 0.443, Metric: 0.471
Epoch 14/20, Loss: 0.421, Metric: 0.468
Epoch 15/20, Loss: 0.559, Metric: 0.541
Epoch 16/20, Loss: 0.486, Metric: 0.505
Epoch 17/20, Loss: 0.448, Metric: 0.485
Epoch 18/20, Loss: 0.419, Metric: 0.466
Epoch 19/20, Loss: 0.412, Metric: 0.463
Epoch 20/20, Loss: 0.41, Metric: 0.46
